# Volume 2, Appendix M: The Linear Algebra Method (Python companion)

Exercises first, solutions at the end. The chapter's theorems are upper bounds proved by dimension counting; here we look for the extremal objects by search and see how close they come: Oddtown and Eventown clubs, Fisher families, two-distance sets, the Frankl–Wilson graphs, sumsets modulo a prime, zero-sum subsets, cap sets in SET, and small Kakeya sets.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
random.seed(11); rng = np.random.default_rng(11)

## Exercises

**P11.1 (★) Oddtown and Eventown by search.** For $n\le6$, find the largest Oddtown and Eventown families by a maximum clique search, and compare with $n$ and $2^{\lfloor n/2\rfloor}$ (Theorems 11.1 and 11.3). Check that the $16$ clubs of Exercise 11.1 are an Eventown family.

**P11.2 (★) Fisher and two distances.** Check Fisher's inequality (Theorem 11.4) on all families of subsets of $[5]$ with constant pairwise intersection $\lambda=1$ found by clique search. Check numerically that the pentagon and the simplex midpoints (Exercise 11.3) are two-distance sets, and tabulate the bounds of Theorem 11.5 against the construction.

**P11.3 (★★) Frankl–Wilson graphs.** Build the graph of Theorem 11.8 for the small parameters ($k$-subsets of $[m]$, adjacent when $|A\cap B|\not\equiv-1\pmod p$) with $p=2$ and $p=3$ on a reduced ground set, and compute the largest clique and independent set exactly. Compare with the bound $\sum_{i<p}\binom mi$.

**P11.4 (★) Cauchy–Davenport and Erdős–Ginzburg–Ziv.** Check Theorem 11.11 on all pairs of subsets of $\Z/7\Z$, find pairs where it fails in $\Z/8\Z$, and check Theorem 11.14 exhaustively for small $n$.

**P11.5 (★) Chevalley–Warning.** Count the common zeros of random systems over $\mathbb F_p$ with $\sum\deg<n$ and check that the counts are divisible by $p$; then see what happens when $\sum\deg\ge n$.

**P11.6 (★★) Caps.** Find the largest caps in $\mathbb F_3^n$ for $n\le3$ by exhaustive search, and look for large caps in $\mathbb F_3^4$ (the SET deck) by randomised search. Compare with $3M_n$ (Theorem 11.19) and $3^n$.

**P11.7 (★★) Slice rank numerics.** Compute $M_n$ exactly, compare $(3M_n)^{1/n}$ with the constant $2.7551$ of Proposition 11.20, and plot the minimisation over $x$.

**P11.8 (★) Kakeya sets.** Find the smallest Kakeya sets in $\mathbb F_3^2$ and $\mathbb F_5^2$ by search, and compare with Dvir's bound $\binom{q+1}2$ (Exercise 11.10).

## Solutions

### P11.1 Oddtown and Eventown by search

A family satisfying the rules is a clique in the graph whose vertices are the allowed sets (odd, or even), joined when their intersection is even. Networkx finds maximum cliques exactly at this size.

In [ ]:
def max_family(n, parity):
    sets = [frozenset(S) for k in range(n + 1) for S in itertools.combinations(range(n), k) if k % 2 == parity]
    G = nx.Graph(); G.add_nodes_from(sets)
    G.add_edges_from((A, B) for A, B in itertools.combinations(sets, 2) if len(A & B) % 2 == 0)
    clique, size = nx.max_weight_clique(G, weight=None)
    return size, clique
for n in range(1, 7):
    o, _ = max_family(n, 1); e, fam = max_family(n, 0)
    print(f'n={n}: Oddtown max {o} (bound {n}),  Eventown max {e} (bound {2 ** (n // 2)})')

In [ ]:
pts = list(itertools.product(range(2), repeat=3))
planes = set()
for normal in pts[1:]:
    for c in range(2):
        planes.add(frozenset(p for p in pts if sum(a * b for a, b in zip(normal, p)) % 2 == c))
clubs = list(planes) + [frozenset(), frozenset(pts)]
print(len(clubs), 'clubs; sizes', sorted(Counter(len(A) for A in clubs).items()),
      '; all intersections even:', all(len(A & B) % 2 == 0 for A, B in itertools.combinations(clubs, 2)))

Search confirms that the bounds are exact for every $n\le6$. The $16$ clubs on the points of $\mathbb F_2^3$ (sizes $0$, $4$, $8$) are an Eventown family of maximum size with no club of size $2$.

### P11.2 Fisher and two distances

In [ ]:
n = 5
sets = [frozenset(S) for k in range(1, n + 1) for S in itertools.combinations(range(n), k)]
G = nx.Graph(); G.add_nodes_from(sets)
G.add_edges_from((A, B) for A, B in itertools.combinations(sets, 2) if len(A & B) == 1)
best = max(nx.find_cliques(G), key=len)
print('largest family of subsets of [5] with all pairwise intersections of size 1:', len(best),
      sorted(sorted(A) for A in best))

The maximum is $5=n$, attained by the near-pencil (Exercise 11.2). Now two-distance sets.

In [ ]:
def distances(P, digits=9):
    return sorted({round(float(np.linalg.norm(np.array(a) - np.array(b))), digits) for a, b in itertools.combinations(P, 2)})
pent = [(math.cos(2 * math.pi * k / 5), math.sin(2 * math.pi * k / 5)) for k in range(5)]
print('pentagon distances:', distances(pent), ' ratio', distances(pent)[1] / distances(pent)[0], '(the golden ratio)')
for n in range(2, 9):
    P = [tuple(1 if t in (i, j) else 0 for t in range(n + 1)) for i, j in itertools.combinations(range(n + 1), 2)]
    print(f'R^{n}: simplex midpoints {len(P):3d} points, distances {distances(P)};  bounds: Blokhuis {(n+1)*(n+2)//2:3d}, LRS {(n+1)*(n+4)//2:3d}')

For $n=8$ the best known two-distance set has $45$ points (Lisoněk, 1997), better than the simplex construction's $36$ and close to Blokhuis' $45$: in dimension $8$ the bound is exact.

### P11.3 Frankl–Wilson graphs

The true graph of Theorem 11.8 for $p=3$ has two million vertices, too many for exact clique search. The proof works for $k$-subsets of $[m]$ with $k\equiv-1\pmod p$ and any $m$ (the independent sets have intersections in $\{p-1,2p-1,\dots\}$ below $k$, at most $\lfloor k/p\rfloor$ values), so we look at small cases and compare the exact answer with the bound.

In [ ]:
def fw_graph(m, k, p):
    V = [frozenset(S) for S in itertools.combinations(range(m), k)]
    G = nx.Graph(); G.add_nodes_from(V)
    G.add_edges_from((A, B) for A, B in itertools.combinations(V, 2) if len(A & B) % p != p - 1)
    return G
for m, k, p in [(6, 3, 2), (7, 3, 2), (8, 3, 2), (7, 5, 3), (8, 5, 3)]:
    G = fw_graph(m, k, p)
    om = nx.max_weight_clique(G, weight=None)[1]
    al = nx.max_weight_clique(nx.complement(G), weight=None)[1]
    s_ind = len([t for t in range(p - 1, k, p)])
    print(f'm={m}, k={k}, p={p}: {G.number_of_nodes():3d} vertices, clique {om} (bound {sum(math.comb(m, i) for i in range(p))}), '
          f'independent {al} (bound {sum(math.comb(m, i) for i in range(s_ind + 1))})')

The bounds hold, with room to spare at this size; the point of the construction is the growth: for $m=p^3$ and $k=p^2-1$ the number of vertices outruns every power of the bound. For $p=3$: $\binom{27}8=2\,220\,075$ vertices, and no clique or independent set larger than $379$.

### P11.4 Cauchy–Davenport and Erdős–Ginzburg–Ziv

In [ ]:
def sumset(A, B, n):
    return {(a + b) % n for a in A for b in B}
def subsets(n):
    return [S for k in range(1, n + 1) for S in itertools.combinations(range(n), k)]
p = 7
assert all(len(sumset(A, B, p)) >= min(p, len(A) + len(B) - 1) for A in subsets(p) for B in subsets(p))
print('Cauchy-Davenport holds for all', len(subsets(p)) ** 2, 'pairs in Z/7')
bad = [(A, B) for A in subsets(8) for B in subsets(8) if len(sumset(A, B, 8)) < min(8, len(A) + len(B) - 1)]
print('Z/8: failures', len(bad), '; for example', bad[0], '->', sorted(sumset(*bad[0], 8)))

In [ ]:
def egz_holds(seq, n):
    return any(sum(c) % n == 0 for c in itertools.combinations(seq, n))
for n in range(2, 7):
    ok = all(egz_holds(seq, n) for seq in itertools.combinations_with_replacement(range(n), 2 * n - 1))
    sharp = not egz_holds([0] * (n - 1) + [1] * (n - 1), n)
    print(f'n={n}: every multiset of {2*n-1} residues has a zero-sum n-subset: {ok};  0^(n-1) 1^(n-1) has none: {sharp}')

Residues suffice, and multisets of residues suffice, so the check is exhaustive. In $\Z/8\Z$ the failures all come from cosets of subgroups, like $\{0,4\}+\{0,4\}=\{0,4\}$.

### P11.5 Chevalley–Warning

In [ ]:
def rand_poly(n, d, p):
    # random polynomial of degree <= d with no constant term, as a list of (coef, exponent tuple)
    mons = [e for e in itertools.product(range(d + 1), repeat=n) if 0 < sum(e) <= d]
    return [(random.randrange(p), e) for e in random.sample(mons, min(len(mons), 6))]
def ev(P, x, p):
    return sum(c * math.prod(xi ** ei for xi, ei in zip(x, e)) for c, e in P) % p
def zeros(Ps, n, p):
    return sum(all(ev(P, x, p) == 0 for P in Ps) for x in itertools.product(range(p), repeat=n))
for p, n, degs in [(3, 4, [2, 1]), (5, 3, [2]), (3, 5, [2, 2]), (2, 6, [3, 2]), (3, 3, [2, 1]), (5, 2, [2])]:
    counts = [zeros([rand_poly(n, d, p) for d in degs], n, p) for _ in range(6)]
    tag = 'sum deg < n' if sum(degs) < n else 'sum deg >= n'
    print(f'p={p}, n={n}, degrees {degs} ({tag}): zero counts {counts}, mod p {[c % p for c in counts]}')

When $\sum\deg<n$ every count is divisible by $p$; with $\sum\deg\ge n$ the residues are arbitrary (for instance $x^2+y^2=0$ over $\mathbb F_5$ has $9$ solutions).

### P11.6 Caps

A backtracking search adds points in increasing order and forbids, after each addition, the third point $-x-y$ of every line through the new point and an earlier one. For $n=3$ we may assume the cap contains $0$ (translate it).

In [ ]:
def all_points(n):
    return list(itertools.product(range(3), repeat=n))
def third(x, y):
    return tuple((-a - b) % 3 for a, b in zip(x, y))
def max_cap(n):
    P = all_points(n); idx = {p: i for i, p in enumerate(P)}; best = []
    def rec(cap, start, forbidden):
        nonlocal best
        if len(cap) > len(best):
            best = cap[:]
        for i in range(start, len(P)):
            if len(cap) + (len(P) - i) <= len(best):
                return
            if i in forbidden:
                continue
            new = {idx[third(P[i], P[j])] for j in cap}
            rec(cap + [i], i + 1, forbidden | new)
    rec([0], 1, set())
    return [P[i] for i in best]
def is_cap(A):
    S = set(A)
    return all(third(x, y) not in S or third(x, y) in (x, y) for x, y in itertools.combinations(A, 2))
for n in range(1, 4):
    c = max_cap(n)
    print(f'n={n}: largest cap {len(c)}, is a cap: {is_cap(c)}')

In [ ]:
def random_cap(n, tries):
    P = all_points(n); best = []
    for _ in range(tries):
        random.shuffle(P); cap, bad = [], set()
        for x in P:
            if x not in bad:
                bad |= {third(x, y) for y in cap}; cap.append(x)
        # local improvement: remove one point, try to add two
        improved = True
        while improved:
            improved = False
            for r in list(cap):
                rest = [y for y in cap if y != r]
                badr = {third(a, b) for a, b in itertools.combinations(rest, 2)} | set(rest)
                cands = [x for x in P if x not in badr]
                for a, b in itertools.combinations(cands, 2):
                    if third(a, b) not in rest and a != b and is_cap(rest + [a, b]):
                        cap = rest + [a, b]; improved = True; break
                if improved:
                    break
        if len(cap) > len(best):
            best = cap
    return best
cap4 = random_cap(4, 60)
print('F_3^4 (the SET deck): found a cap of', len(cap4), 'cards; is a cap:', is_cap(cap4))

The search reliably reaches $20$, Pellegrino's maximum: $20$ SET cards with no set among them. Proving that $21$ is impossible by brute force is another matter; it is a classical but delicate counting argument (see *The Joy of SET*). Compare with the bounds below.

### P11.7 Slice rank numerics

In [ ]:
def M(n):
    # number of a in {0,1,2}^n with sum <= 2n/3, by dynamic programming on the sum
    cnt = [1]
    for _ in range(n):
        new = [0] * (len(cnt) + 2)
        for s, c in enumerate(cnt):
            for a in range(3):
                new[s + a] += c
        cnt = new
    return sum(c for s, c in enumerate(cnt) if 3 * s <= 2 * n)
for n, known in [(1, 2), (2, 4), (3, 9), (4, 20), (5, 45), (6, 112), (10, None), (50, None), (200, None)]:
    b = 3 * M(n)
    print(f'n={n:3d}: max cap {str(known):>4s}   bound 3M_n = {b:.4g}   3^n = {3**n:.4g}   (3M_n)^(1/n) = {b ** (1 / n):.4f}')
xs = np.linspace(0.3, 1, 300); f = (1 + xs + xs ** 2) / xs ** (2 / 3)
x0 = (math.sqrt(33) - 1) / 8
print('optimal x =', round(x0, 4), ' constant =', round((1 + x0 + x0 ** 2) / x0 ** (2 / 3), 5), ' at x = 0.6:', round(1.96 / 0.6 ** (2 / 3), 5))
plt.plot(xs, f); plt.axvline(x0, ls=':'); plt.axhline(3, ls='--', c='gray')
plt.xlabel('x'); plt.ylabel('(1+x+x^2)/x^(2/3)'); plt.title('the Chernoff-type bound on M_n^(1/n)'); plt.show()

For $n\le3$ the bound $3M_n$ is no better than the trivial $3^n$, and for small $n$ it is far above the true maxima. As $n$ grows, $(3M_n)^{1/n}$ approaches $2.7551$ slowly from below (the factor $3$ and the polynomial factors hidden in Proposition 11.20 fade like $c^{1/n}$), while $3^n$ grows exponentially faster. At $x=1$ the bound is the trivial $3$; the minimum is at $x=\frac{\sqrt{33}-1}8$.

### P11.8 Kakeya sets

In [ ]:
def is_kakeya(K, q):
    S = set(K)
    dirs = [(1, m) for m in range(q)] + [(0, 1)]
    return all(any(all(((x + t * d[0]) % q, (y + t * d[1]) % q) in S for t in range(q))
                   for x in range(q) for y in range(q)) for d in dirs)
def min_kakeya(q):
    P = list(itertools.product(range(q), repeat=2))
    # a Kakeya set is a union of q+1 lines, one per direction: search over choices of lines
    dirs = [(1, m) for m in range(q)] + [(0, 1)]
    lines = [[frozenset(((x + t * d[0]) % q, (y + t * d[1]) % q) for t in range(q)) for x, y in P] for d in dirs]
    lines = [list(set(L)) for L in lines]
    best = None
    for choice in itertools.product(*lines):
        U = frozenset().union(*choice)
        if best is None or len(U) < len(best):
            best = U
    return best
for q in [3, 5]:
    K = min_kakeya(q)
    print(f'q={q}: smallest Kakeya set in F_q^2 has {len(K)} points (Dvir: >= {math.comb(q + 1, 2)}; whole plane {q*q}); check {is_kakeya(K, q)}')

In the plane the true minimum is $\frac{q(q+1)}2+\frac{q-1}2$ for odd $q$ (Blokhuis and Mazzocca, 2008), so Dvir's bound $\binom{q+1}2$ is off only by $\frac{q-1}2$. The search over one line per direction is exhaustive because a minimal Kakeya set is a union of $q+1$ lines.